# 2.5 보상 함수

PBRFT는 최종 점수 하나만 사용하지만, Agentic RL은 다단계 보상 신호를 결합한다.

**핵심**: 희소 보상(종료) + 밀집 보상(과정) + 보조 신호(도구 효율, 안전)



### PBRFT: 단일 종료 보상
R_final: 응답에 대한 선호도 점수 (1회)

### Agentic RL: 다단계 보상
R_t = R_task(t) + λ_process × R_process(t) + λ_tool × R_tool(t) + λ_safety × R_safety(t)

### Generative Agent Memory와의 연결
'Reflect' 컴포넌트가 과거 경험을 분석하여 내부 보상 신호 생성한다.

llm_reward() 함수를 사용하여 LLM을 판사(judge)로 활용한다.

In [1]:
from utils_openai import *

## 실습 1: PBRFT의 단일 보상

In [2]:
heading("PBRFT: 단일 종료 보상")

prompt = "2+2는 몇인가?"
response = ask(prompt, temperature=0.0)

step_print(1, "프롬프트", prompt)
step_print(2, "응답", response)

# 최종 보상만 계산 (LLM 판사 사용)
criteria = "수학 답의 정확성"
final_reward = llm_reward(response, criteria=criteria, max_score=10)

kv_print([("최종 보상", f"{final_reward:.1f}/10 (종료 신호)")])
print("\
✓ PBRFT는 마지막에 한 번의 평가만 받는다.")


────────────────────────────────────────
  PBRFT: 단일 종료 보상
────────────────────────────────────────
  [Step 1] 프롬프트
    2+2는 몇인가?
  [Step 2] 응답
    2 + 2는 4입니다.
  최종 보상  10.0/10 (종료 신호)
✓ PBRFT는 마지막에 한 번의 평가만 받는다.


## 실습 2: Agentic RL의 다단계 보상

In [3]:
heading("Agentic RL: 다단계 보상")

memory = MemoryStream()
trajectory = []

# 단계별 보상
steps_info = [
    {"action": "분석", "reward_task": 1.0, "reward_process": 0.5},
    {"action": "계산", "reward_task": 3.0, "reward_process": 1.0},
    {"action": "검증", "reward_task": 6.0, "reward_process": 1.5}
]

total_reward = 0.0
for i, step in enumerate(steps_info):
    # 다단계 보상 계산
    task_reward = step["reward_task"]
    process_reward = step["reward_process"]
    combined = task_reward + 0.5 * process_reward  # 가중치 적용
    total_reward += combined
    
    step_print(i+1, f"Step {i}: {step['action']}",
               f"Task={task_reward:.1f} + Process={process_reward:.1f} = {combined:.1f}")
    trajectory.append(combined)

# 할인 누적
gamma = 0.9
discounted = sum(r * (gamma ** i) for i, r in enumerate(trajectory))

kv_print([
    ("누적 보상", f"{total_reward:.1f}"),
    ("할인 누적 (γ=0.9)", f"{discounted:.2f}")
])
print("\
✓ Agentic RL은 각 단계마다 피드백을 받으며 진행한다.")



────────────────────────────────────────
  Agentic RL: 다단계 보상
────────────────────────────────────────
  [Step 1] Step 0: 분석
    Task=1.0 + Process=0.5 = 1.2
  [Step 2] Step 1: 계산
    Task=3.0 + Process=1.0 = 3.5
  [Step 3] Step 2: 검증
    Task=6.0 + Process=1.5 = 6.8
  누적 보상          11.5
  할인 누적 (γ=0.9)  9.87
✓ Agentic RL은 각 단계마다 피드백을 받으며 진행한다.


## 실습 3: llm_reward()로 LLM 판사 사용하기

In [4]:
heading("LLM 기반 보상 평가")

responses = [
    "2+2는 4입니다.",
    "2+2는 5입니다.",
    "2+2는 4이며, 이는 2개의 2를 더한 결과입니다."
]

print("각 응답의 보상 점수 (LLM 판사):")
for i, resp in enumerate(responses):
    score = llm_reward(resp, criteria="수학 정확성", max_score=10)
    step_print(i+1, f"응답 {i+1}", f"{resp[:40]}... → 점수 {score:.1f}")

print("\
✓ LLM을 판사로 사용하여 정교한 평가가 가능하다.")



────────────────────────────────────────
  LLM 기반 보상 평가
────────────────────────────────────────
각 응답의 보상 점수 (LLM 판사):
  [Step 1] 응답 1
    2+2는 4입니다.... → 점수 10.0
  [Step 2] 응답 2
    2+2는 5입니다.... → 점수 0.0
  [Step 3] 응답 3
    2+2는 4이며, 이는 2개의 2를 더한 결과입니다.... → 점수 10.0
✓ LLM을 판사로 사용하여 정교한 평가가 가능하다.


## 핵심 정리

| 측면 | PBRFT | Agentic RL |
|------|-------|----------|
| **보상** | 단일 (최종) | 다단계 (과정 + 결과) |
| **평가** | 1회 | 각 단계마다 |
| **신호** | 희소 | 희소 + 밀집 + 보조 |
| **신용 할당** | 직접 | 복잡 (시간적 의존성) |
| **도구** | llm_reward() | llm_reward() + 커스텀 |